In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage


In [4]:

load_dotenv()
LOGINOM_URL = "https://edu.loginom.dev/lgi/rest/instacart_ws_Abakarov1/GetUserHistory"
USER_ID = 1

payload = {
    "Variables": {
        "user_id": USER_ID
    }
}


response = requests.post(LOGINOM_URL, json=payload, timeout=30)
response.raise_for_status()
rows = response.json()["DataSet"]["Rows"]

# преобразуем ответ сервиса в датафрейм
df = pd.DataFrame(rows)
print(df.head(10))


                       product_name                   aisle  department  \
0               Original Beef Jerky           popcorn jerky      snacks   
1                              Soda             soft drinks   beverages   
2                        Pistachios  nuts seeds dried fruit      snacks   
3             Organic String Cheese         packaged cheese  dairy eggs   
4             Cinnamon Toast Crunch                  cereal   breakfast   
5                 Zero Calorie Cola             soft drinks   beverages   
6        Aged White Cheddar Popcorn           popcorn jerky      snacks   
7            Bag of Organic Bananas            fresh fruits     produce   
8               Organic Half & Half                   cream  dairy eggs   
9  XL Pick-A-Size Paper Towel Rolls             paper goods   household   

            department_rus  order_count  
0                    снэки           10  
1            напитки, соки           10  
2                    снэки            9  
3  мол

In [ ]:


lines = []
for i, row in enumerate(rows, start=1):
    lines.append(
        f"{i}. {row['product_name']} — "
        f"отдел: {row['department_rus']}, "
        f"категория: {row['aisle']}, "
        f"заказов: {row['order_count']}"
    )

products_text = "\n".join(lines)
print(products_text)


1. Original Beef Jerky — отдел: снэки, категория: popcorn jerky, заказов: 10
2. Soda — отдел: напитки, соки, категория: soft drinks, заказов: 10
3. Pistachios — отдел: снэки, категория: nuts seeds dried fruit, заказов: 9
4. Organic String Cheese — отдел: молочные продукты, яйца, категория: packaged cheese, заказов: 8
5. Cinnamon Toast Crunch — отдел: сухие завтраки, категория: cereal, заказов: 3
6. Zero Calorie Cola — отдел: напитки, соки, категория: soft drinks, заказов: 3
7. Aged White Cheddar Popcorn — отдел: снэки, категория: popcorn jerky, заказов: 2
8. Bag of Organic Bananas — отдел: овощи, фрукты, зелень, категория: fresh fruits, заказов: 2
9. Organic Half & Half — отдел: молочные продукты, яйца, категория: cream, заказов: 2
10. XL Pick-A-Size Paper Towel Rolls — отдел: товары для дома, категория: paper goods, заказов: 2


In [ ]:

api_key = os.getenv("GIGA_KEY")
if not api_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

llm = GigaChat(
    credentials=api_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.3,
    max_tokens=500
)


In [11]:


system_prompt = """Ты — персональный помощник покупателя в продуктовом онлайн-магазине.
Твоя задача — анализировать историю покупок клиента и давать ему понятные,
дружелюбные рекомендации прямо в его личном кабинете.
Обращайся к клиенту на "вы", тепло и по-человечески.
Пиши кратко — не более 5–6 предложений."""

user_prompt = f"""Вот список товаров, которые клиент покупает чаще всего:

{products_text}

Напишите клиенту короткий персональный анализ его покупок.
Начните с того, что вы заметили в его предпочтениях.
Дайте 1–2 дружеских совета — например, что стоит попробовать добавить
или на что обратить внимание. Пишите так, как будто клиент читает это
в своём личном кабинете."""


In [12]:


messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_prompt)
]


result = llm.invoke(messages)
print(result.content)


ConnectTimeout: timed out